In [10]:
import os
import sys
import json

# Add the project root to sys.path
# sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import openai
import pandas as pd

# from dotenv import load_dotenv
# load_dotenv()

from src.api.core.config import config
from src.api.rag.retrieval import rag_pipeline

from langsmith import Client
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper




#### Initiate Langsmith & Qdrant client

In [4]:

ls_client = Client(api_key=config.LANGSMITH_API_KEY)

In [5]:
# qdrant_client = QdrantClient(
#     url=f"http://localhost:6333"
# )
qdrant_client = QdrantClient(
    url=config.QDRANT_URL,
    api_key=config.QDRANT_API_KEY  # For Qdrant Cloud only
)

#### Sample 50 items from the dataset

In [6]:
# Collect some chunks from your DB (example: first 50)
# If you already have them in memory, just pass that list instead
all_chunks = qdrant_client.scroll(
    collection_name=config.QDRANT_COLLECTION_NAME,
    limit=50
)[0]

all_chunks

[Record(id='0017143e-deec-4865-83da-4b55124cd615', payload={'file_name': 'Smith_2024_ApJ_974_292.pdf', 'file_hash': '5cde3e39b7ce12f03cf037547383b67a3395ff0d18fac94b2fc549ecfb4bb9b2', 'file_title': 'Variable Stars in M31 Stellar Clusters from the Panchromatic Hubble Andromeda Treasury', 'authors': ['Léo Girardi', 'Avi Patel', 'Sagnick Mukherjee', 'Benjamin F. Williams', 'Pranav Tadepalli', 'Richard Smith', 'Puragra Guhathakurta', 'L. Clifton Johnson', 'Sally Zhu', 'Joseph Liu', 'Monika D. Soraisam', 'Knut A. G. Olsen'], 'keywords': ['Hubble Space Telescope', 'M31', 'Variable stars', 'Star clusters', 'Panchromatic Hubble Andromeda Treasury', 'Andromeda Galaxy', 'Stellar clusters'], 'creation_date': "D:20241017175216+05'30'", 'year': '2024', 'page_number': '27', 'text': 'measurements affected by cosmic-ray strikes, bad pixels, etc. as described in Section 3.1.20 The phot_mod table can be queried by\nchanging the phot_meas entry in the code block, along with any relevant table column head

In [7]:
data_to_embed = [point.payload["text"] for point in all_chunks]

data_to_embed

['measurements affected by cosmic-ray strikes, bad pixels, etc. as described in Section 3.1.20 The phot_mod table can be queried by\nchanging the phot_meas entry in the code block, along with any relevant table column headers speciﬁed in the query.\nphat_id = ”PHAT_10.9517775+41.448073”\nquery_all = ”””SELECT *\nFROM phat_v2.phot_meas\nWHERE objid = ’%s’ AND ﬁlter = ’F814W ’ AND magvega < 99 .0 and sharp ^2 < 0.2\n”””%phat_id\ntry:\nresult1 = qc.query(auth_token, sql = query_all, timeout = 400)\nexcept Exception as e:\nprint(e)\ndf_all = helpers.utils.convert(result1,’pandas’)\nquery_cols = ”””SELECT mjd, magvega, magerr,sharp\nFROM phat_v2.phot_meas\nWHERE objid = ’%s’ AND ﬁlter = ’F814W ’ AND magvega < 99 .0 and sharp ^2 < 0.2\n”””%phat_id\ntry:\nresult2 = qc.query(auth_token, sql = query_cols, timeout = 400)\nexcept Exception as e:\nprint(e)\ndf_cols = helpers.utils.convert(result2,’pandas’)\nAppendix D\nForeground Star Identiﬁcation\nWe cross-match our initial sample of 376 luminou

#### Render a prompt to generate synthetic Eval reference dataset

In [11]:
# import json

# output_schema = {
#     "type": "array",
#     "items": {
#         "type": "object",
#         "properties": {
#             "question": {
#                 "type": "string",
#                 "description": "Suggested question.",
#             },
#             "chunk_ids": {
#                 "type": "array",
#                 "items": {
#                     "type": "integer",
#                     "description": "Index of the chunk that could be used to answer the question.",
#                 },
#             },
#             "answer_example": {
#                 "type": "string",
#                 "description": "Suggested answer grounded in the contexr.",
#             },
#             "reasoning": {
#                 "type": "string",
#                 "description": "Reasoning why the question could be answered with the chunks.",
#             },
#         },
#     },
# }


# SYSTEM_PROMPT = f"""
# I am building a RAG application. I have a collection of 50 chunks of text.
# The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
# I will provide all of the available products to you with indexes of each chunk.
# I want you to come up with 30 questions to which the answers could be grounded in the chunk context.
# As an output I need you to provide me the list of questions and the indexes of the chunks that could be used to answer them.
# Also, provide an example answer to the question given the context of the chunks.
# Also, provide the reason why you chose the chunks to answer the questions.
# Try to have a mix of questions that could use multipple chunks and questions that could use single chunk.
# Also, include 5 questions that can't be answered with the available chunks.

# <OUTPUT JSON SCHEMA>
# {json.dumps(output_schema, indent=2)}
# </OUTPUT JSON SCHEMA>

# I need to be able to parse the json output.
# """

# USER_PROMPT = f"""
# Here is the list of chunks, each list element is a dictionary with id and text:
# {[{"id": i, "text": data} for i, data in enumerate(data_to_embed)]}
# """


# JSON schema for the dataset
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "integer",
                    "description": "Indexes of the chunks used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Ground-truth answer based only on the chunks.",
            },
            "reasoning": {
                "type": "string",
                "description": "Why these chunks support the answer.",
            },
        },
    },
}

SYSTEM_PROMPT = f"""
I am building a Retrieval-Augmented Generation (RAG) application. 
It should answer questions about scientific literature from provided chunks. 

You will receive a list of chunks (with IDs and text).
Your task is to generate 30 evaluation questions.

Guidelines:
- Questions must be grounded in the content of the chunks.
- Provide a diverse set of questions (factual, multi-hop, entity-based, etc).
- At least 5 questions should be unanswerable with the given chunks.
- For each question, provide:
  * The question itself
  * The IDs of chunks that contain the answer
  * An example answer (based only on those chunks)
  * A short reasoning why those chunks support the answer

Return ONLY valid JSON following this schema:

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>
"""

USER_PROMPT = f"""
Here is the list of chunks:
{[{"id": i, "text": data} for i, data in enumerate(data_to_embed)]}
"""



#### Generate synthetic eval reference data

In [12]:
response = openai.chat.completions.create(
    model="gpt-4.1",
    temperature=0.7,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ]
)

raw_output = response.choices[0].message.content

# Save to JSON
dataset = json.loads(raw_output)
with open("rag_evaluation_dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

#### Clean up the output and make it a parseable json

In [13]:
import json

json_output = response.choices[0].message.content
json_output = json_output.replace("```json", "")
json_output = json_output.replace("```", "")
json_output = json_output.replace("// BEGIN UNANSWERABLE QUESTIONS SECTION (5)", "")
json_output = json.loads(json_output)

json_output

[{'question': 'What is the main advantage of using stochastic models over traditional Simple Stellar Population (SSP) models when deriving physical parameters of low-mass star clusters?',
  'chunk_ids': [26, 2, 25],
  'answer_example': 'Stochastic models take into account the random sampling of the stellar initial mass function (IMF), which leads to a more realistic dispersion of integrated cluster colors. This avoids strong biases present in SSP models, which assume continuous IMF sampling and can misestimate parameters like age and mass, especially for low-mass clusters.',
  'reasoning': 'Chunk 26 (Abstract) and chunk 2 discuss the limitations of SSP models and the benefits of stochastic models for realistic parameter derivation. Chunk 25 further details the importance of accounting for stochasticity in low-mass clusters.'},
 {'question': 'Which photometric system provides the highest accuracy for deriving metallicity in unresolved star clusters, and under what conditions?',
  'chunk

#### Upload the dataset to LangSmith

In [14]:
from langsmith import Client
import os

client = Client(api_key=os.environ["LANGSMITH_API_KEY"])

dataset_name = "rag-evaluation-dataset"
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Dataset for evaluating RAG pipeline"
)

In [16]:
records = qdrant_client.retrieve(
    collection_name=config.QDRANT_COLLECTION_NAME,
    ids=item["chunk_ids"],
    with_payload=True
)
contexts = [rec.payload["text"] for rec in records]

In [17]:
# for item in json_output:
#     client.create_example(
#         dataset_id=dataset.id,
#         inputs={"question": item["question"]},
#         outputs={
#             "ground_truth": item["answer_example"],
#             "context_ids": item["chunk_ids"],
#             "contexts": [qdrant_client.retrieve(collection_name=config.QDRANT_COLLECTION_NAME, ids=[id], with_payload=True)[0].payload["text"] for id in item["chunk_ids"]]
#         }
#     )

for item in json_output:
    records = qdrant_client.retrieve(
        collection_name=config.QDRANT_COLLECTION_NAME,
        ids=item["chunk_ids"],
        with_payload=True
    )
    contexts = [rec.payload["text"] for rec in records]

    client.create_example(
        dataset_id=dataset.id,
        inputs={"question": item["question"]},
        outputs={
            "ground_truth": item["answer_example"],
            "context_ids": item["chunk_ids"],
            "contexts": contexts
        }
    )
